# PDE vs ODE Comparison

Validate the ODE approximation by comparing against the exact PDE solution
across the parameter ranges used in the sensitivity analysis.

**Compute phase** (cells 1-3): Run PDE solves in parallel, save to `data/pde_comparison/`.
Skip these cells after the first run — all subsequent cells load from disk.

**Analysis phase** (cells 4+): Load cached results and produce figures/tables.

In [9]:
import os
import json
import time
import numpy as np
import matplotlib.pyplot as plt
from multiprocessing import Pool
from scipy.stats import spearmanr

from src.pde_comparison import (
    PARAM_LABELS, NOMINAL, RANGES, N_PARAMS,
    QOI_NAMES, QOI_LABELS, N_QOIS,
    build_modified_params_2ccy, evaluate_ode_qois, pde_extract_qois,
    run_single_pde, _init_worker,
    generate_oat_configs, generate_lhs_configs,
    Y_MAX, DY, PDE_N_STEPS,
)
from src import (
    build_paper_example_params, restrict_currencies,
    run_multicurrency_mm, BP,
)

SAVE_DIR = "data/pde_comparison"
N_WORKERS = 10

In [10]:
oat_configs = generate_oat_configs()
lhs_configs = generate_lhs_configs(n_samples=20, seed=42)
all_configs = oat_configs + lhs_configs

print(f"OAT configs: {len(oat_configs)} (1 nominal + {N_PARAMS} params x 2 edges)")
print(f"LHS configs: {len(lhs_configs)}")
print(f"Total PDE solves: {len(all_configs)}")

OAT configs: 17 (1 nominal + 8 params x 2 edges)
LHS configs: 20
Total PDE solves: 37


In [11]:
os.makedirs(os.path.join(SAVE_DIR, "oat"), exist_ok=True)
os.makedirs(os.path.join(SAVE_DIR, "lhs"), exist_ok=True)

t0 = time.time()

# Use imap_unordered for tqdm progress tracking
from tqdm.notebook import tqdm

with Pool(processes=N_WORKERS, initializer=_init_worker) as pool:
    results = []
    for res in tqdm(pool.imap_unordered(run_single_pde, all_configs),
                    total=len(all_configs), desc="PDE solves"):
        results.append(res)
        
        # Save each result immediately
        name = res['name']
        subdir = "lhs" if name.startswith("lhs_") else "oat"
        np.savez_compressed(
            os.path.join(SAVE_DIR, subdir, f"{name}.npz"),
            theta_0=res['theta_0'],
            ode_qois=res['ode_qois'],
            pde_qois=res['pde_qois'],
            params=res['params'],
        )

elapsed = time.time() - t0
print(f"\nCompleted {len(results)} PDE solves in {elapsed/60:.1f} min")
print(f"Average: {elapsed/len(results):.0f} s/solve")

# Save metadata
meta = {
    "n_oat": len(oat_configs),
    "n_lhs": len(lhs_configs),
    "Y_MAX": Y_MAX,
    "DY": DY,
    "PDE_N_STEPS": PDE_N_STEPS,
    "PARAM_LABELS": PARAM_LABELS,
    "QOI_NAMES": QOI_NAMES,
    "NOMINAL": NOMINAL.tolist(),
    "RANGES": RANGES.tolist(),
    "elapsed_seconds": elapsed,
    "n_workers": N_WORKERS,
}
with open(os.path.join(SAVE_DIR, "metadata.json"), "w") as f:
    json.dump(meta, f, indent=2)

print(f"Saved to {SAVE_DIR}/")

ImportError: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html

In [ ]:
# Load all results from disk
def load_results(save_dir):
    """Load all .npz files from oat/ and lhs/ subdirs."""
    oat = {}
    for fn in sorted(os.listdir(os.path.join(save_dir, "oat"))):
        if fn.endswith(".npz"):
            name = fn[:-4]
            data = np.load(os.path.join(save_dir, "oat", fn))
            oat[name] = {k: data[k] for k in data.files}
    lhs = {}
    for fn in sorted(os.listdir(os.path.join(save_dir, "lhs"))):
        if fn.endswith(".npz"):
            name = fn[:-4]
            data = np.load(os.path.join(save_dir, "lhs", fn))
            lhs[name] = {k: data[k] for k in data.files}
    return oat, lhs

oat_data, lhs_data = load_results(SAVE_DIR)
print(f"Loaded {len(oat_data)} OAT + {len(lhs_data)} LHS results")

## Analysis

### Summary error table

In [ ]:
# Build arrays: (n_configs, n_qois) for ODE and PDE
def error_table(data_dict, title):
    """Print error table for a set of configs."""
    print(f"\n{title}")
    print(f"{'Config':>20s}", end="")
    for q in range(N_QOIS):
        print(f"  {QOI_NAMES[q]:>18s}", end="")
    print()
    print(f"{'':>20s}", end="")
    for q in range(N_QOIS):
        print(f"  {'rel err':>18s}", end="")
    print()
    print("-" * (20 + N_QOIS * 20))
    
    for name in sorted(data_dict.keys()):
        d = data_dict[name]
        ode = d['ode_qois']
        pde = d['pde_qois']
        print(f"{name:>20s}", end="")
        for q in range(N_QOIS):
            rel = abs(ode[q] - pde[q]) / (abs(pde[q]) + 1e-30)
            print(f"  {rel:18.2e}", end="")
        print()

error_table(oat_data, "OAT Configurations")

# Summary statistics
print("\n\nSummary across OAT configs:")
print(f"{'QoI':>20s}  {'max rel err':>12s}  {'mean rel err':>12s}  {'max abs err':>12s}")
for q in range(N_QOIS):
    rel_errs = []
    abs_errs = []
    for d in oat_data.values():
        ode, pde = d['ode_qois'][q], d['pde_qois'][q]
        rel_errs.append(abs(ode - pde) / (abs(pde) + 1e-30))
        abs_errs.append(abs(ode - pde))
    print(f"{QOI_NAMES[q]:>20s}  {max(rel_errs):12.2e}  {np.mean(rel_errs):12.2e}  {max(abs_errs):12.4f}")

### Sensitivity rankings and Spearman rho

In [ ]:
# OAT sensitivity: |QoI(high) - QoI(low)| for each parameter
def compute_oat_sensitivities(data):
    """Compute OAT sensitivity magnitudes for ODE and PDE."""
    ode_sens = np.zeros((N_PARAMS, N_QOIS))
    pde_sens = np.zeros((N_PARAMS, N_QOIS))
    
    for i, label in enumerate(PARAM_LABELS):
        lo_key = f"{label}_low"
        hi_key = f"{label}_high"
        if lo_key in data and hi_key in data:
            ode_sens[i] = np.abs(data[hi_key]['ode_qois'] - data[lo_key]['ode_qois'])
            pde_sens[i] = np.abs(data[hi_key]['pde_qois'] - data[lo_key]['pde_qois'])
    
    return ode_sens, pde_sens

ode_sens, pde_sens = compute_oat_sensitivities(oat_data)

# Rankings
print("Sensitivity rankings (1 = most sensitive):\n")
for q in range(N_QOIS):
    ode_rank = np.argsort(-ode_sens[:, q]) + 1  # descending
    pde_rank = np.argsort(-pde_sens[:, q]) + 1
    
    # Convert to rank arrays
    ode_ranks = np.empty(N_PARAMS, dtype=int)
    pde_ranks = np.empty(N_PARAMS, dtype=int)
    ode_ranks[np.argsort(-ode_sens[:, q])] = np.arange(1, N_PARAMS + 1)
    pde_ranks[np.argsort(-pde_sens[:, q])] = np.arange(1, N_PARAMS + 1)
    
    rho, pval = spearmanr(ode_sens[:, q], pde_sens[:, q])
    
    print(f"{QOI_LABELS[q]}  (Spearman rho = {rho:.3f}, p = {pval:.3g})")
    print(f"  {'Parameter':>15s}  {'ODE rank':>10s}  {'PDE rank':>10s}  {'ODE mag':>12s}  {'PDE mag':>12s}")
    for i in np.argsort(-ode_sens[:, q]):
        print(f"  {PARAM_LABELS[i]:>15s}  {ode_ranks[i]:>10d}  {pde_ranks[i]:>10d}  "
              f"{ode_sens[i, q]:12.4g}  {pde_sens[i, q]:12.4g}")
    print()

### Parity plots (ODE vs PDE)

In [ ]:
os.makedirs("report/figures", exist_ok=True)

fig, axes = plt.subplots(1, N_QOIS, figsize=(4 * N_QOIS, 4))

for q in range(N_QOIS):
    ax = axes[q]
    
    # OAT points
    ode_vals_oat = [d['ode_qois'][q] for d in oat_data.values()]
    pde_vals_oat = [d['pde_qois'][q] for d in oat_data.values()]
    ax.scatter(pde_vals_oat, ode_vals_oat, c='steelblue', s=30, label='OAT', zorder=3)
    
    # LHS points
    ode_vals_lhs = [d['ode_qois'][q] for d in lhs_data.values()]
    pde_vals_lhs = [d['pde_qois'][q] for d in lhs_data.values()]
    ax.scatter(pde_vals_lhs, ode_vals_lhs, c='coral', s=20, marker='x', label='LHS', zorder=3)
    
    # 45-degree line
    all_vals = pde_vals_oat + pde_vals_lhs + ode_vals_oat + ode_vals_lhs
    lo, hi = min(all_vals), max(all_vals)
    margin = 0.05 * (hi - lo) if hi > lo else 1.0
    ax.plot([lo - margin, hi + margin], [lo - margin, hi + margin], 'k--', linewidth=0.8, alpha=0.5)
    
    ax.set_xlabel("PDE")
    ax.set_ylabel("ODE")
    ax.set_title(QOI_LABELS[q], fontsize=10)
    ax.legend(fontsize=8)
    ax.set_aspect('equal', adjustable='datalim')

plt.suptitle("ODE vs PDE: Parity Plots", fontsize=13)
plt.tight_layout()
plt.savefig('report/figures/fig_pde_parity.pdf', bbox_inches='tight', dpi=150)
plt.show()

### Value function profile along y_EUR

In [ ]:
# Load nominal theta_0
theta_nom = oat_data['nominal']['theta_0']
y_grid = np.arange(-Y_MAX, Y_MAX + DY, DY)
idx0 = len(y_grid) // 2

# ODE value function along y_EUR (y_USD = 0)
mp_nom = build_modified_params_2ccy(NOMINAL)
res_nom = run_multicurrency_mm(mp_nom, n_steps=500)
A0, B0 = res_nom.A0, res_nom.B0

# theta_ODE = -y^T A y - y^T B - C
# C is not stored, so we align at y=0: theta_ODE(0) = -C, theta_PDE(0) = known
# Compute ODE profile up to additive constant, then shift to match PDE at origin
theta_ode_profile = np.zeros(len(y_grid))
for j, y_eur in enumerate(y_grid):
    y = np.array([0.0, y_eur])
    theta_ode_profile[j] = -(y @ A0 @ y) - (B0 @ y)

# Shift so ODE matches PDE at y=0
theta_pde_profile = theta_nom[idx0, :]  # y_USD = 0, varying y_EUR
offset = theta_pde_profile[idx0] - theta_ode_profile[idx0]
theta_ode_profile += offset

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Left: both profiles
ax1.plot(y_grid, theta_pde_profile, 'b-', linewidth=1.5, label='PDE')
ax1.plot(y_grid, theta_ode_profile, 'r--', linewidth=1.5, label='ODE')
ax1.set_xlabel(r'$y_{\mathrm{EUR}}$ [M\$]')
ax1.set_ylabel(r'$\theta(0, [0, y_{\mathrm{EUR}}])$')
ax1.set_title('Value function at nominal parameters')
ax1.legend()
ax1.set_xlim(-100, 100)

# Right: difference
diff = theta_pde_profile - theta_ode_profile
ax2.plot(y_grid, diff, 'k-', linewidth=1.5)
ax2.set_xlabel(r'$y_{\mathrm{EUR}}$ [M\$]')
ax2.set_ylabel(r'$\theta_{\mathrm{PDE}} - \theta_{\mathrm{ODE}}$')
ax2.set_title('PDE minus ODE (aligned at y=0)')
ax2.axhline(0, color='grey', linewidth=0.5)
ax2.set_xlim(-100, 100)

plt.tight_layout()
plt.savefig('report/figures/fig_pde_theta_profile.pdf', bbox_inches='tight', dpi=150)
plt.show()

# Report max difference
mask = (y_grid >= -100) & (y_grid <= 100)
print(f"Max |theta_PDE - theta_ODE| in [-100, 100]: {np.max(np.abs(diff[mask])):.2e}")
print(f"Max relative diff: {np.max(np.abs(diff[mask])) / np.max(np.abs(theta_pde_profile[mask])):.2e}")

### Hessian A comparison

In [ ]:
# Extract PDE Hessian at y=0 via central finite differences
# theta_ansatz = -y^T A y - y^T B - C
# d^2 theta / dy_EUR^2 = -2 * A[EUR, EUR]
# So A_PDE[EUR,EUR] = -(1/2) * d^2 theta / dy_EUR^2 at y=0

theta_nom = oat_data['nominal']['theta_0']
idx0 = len(y_grid) // 2

# Second derivative along EUR axis at (y_USD=0, y_EUR=0)
d2_eur = (theta_nom[idx0, idx0+1] - 2*theta_nom[idx0, idx0] + theta_nom[idx0, idx0-1]) / DY**2
A_pde_eur_eur = -0.5 * d2_eur

# ODE A matrix
A_ode = res_nom.A0
A_ode_eur_eur = A_ode[1, 1]  # EUR is index 1

print("Riccati matrix A comparison at y=0:")
print(f"  A_ODE[EUR,EUR] = {A_ode_eur_eur:.6e}")
print(f"  A_PDE[EUR,EUR] = {A_pde_eur_eur:.6e}")
print(f"  Relative diff  = {abs(A_ode_eur_eur - A_pde_eur_eur) / abs(A_pde_eur_eur):.2e}")

# Also extract B from first derivative at y=0
# d theta / dy_EUR at y=0 = -(2*A*y + B)[EUR] = -B[EUR] (since y=0)
d1_eur = (theta_nom[idx0, idx0+1] - theta_nom[idx0, idx0-1]) / (2*DY)
B_pde_eur = -d1_eur
B_ode_eur = res_nom.B0[1]

print(f"\n  B_ODE[EUR]     = {B_ode_eur:.6e}")
print(f"  B_PDE[EUR]     = {B_pde_eur:.6e}")
print(f"  Relative diff  = {abs(B_ode_eur - B_pde_eur) / (abs(B_pde_eur) + 1e-30):.2e}")

### Bridging d=2 to d=3

The sensitivity analysis uses 3 currencies (USD, EUR, GBP) but the PDE
can only validate 2 currencies. The theoretical argument: the ODE error
is per-pair (quadratic Hamiltonian approximation), while cross-pair
interactions are exact in the Riccati matrix algebra.

Below we provide empirical support:
1. Inspect the 3-currency Riccati matrix A(0) -- how large are cross-pair entries?
2. Compare EUR/USD QoIs from 2-currency vs 3-currency ODE -- how much does adding GBP change results?

In [ ]:
from src import build_paper_example_params, restrict_currencies, run_multicurrency_mm

# 3-currency ODE at nominal
mp_3ccy = build_paper_example_params()
mp_3ccy_restricted = restrict_currencies(mp_3ccy, ["USD", "EUR", "GBP"])
res_3ccy = run_multicurrency_mm(mp_3ccy_restricted, n_steps=500)

# 2-currency ODE at nominal (already computed)
res_2ccy = res_nom

print("=== Riccati matrix A(0) -- 3-currency model ===")
ccy = mp_3ccy_restricted.currencies  # ["USD", "EUR", "GBP"]
print(f"Currencies: {ccy}")
print(f"\nA(0) matrix:")
for i, ci in enumerate(ccy):
    row = "  ".join(f"{res_3ccy.A0[i,j]:10.4e}" for j in range(3))
    print(f"  {ci}: {row}")

# Cross-pair coupling ratio
i_eur = ccy.index("EUR")
i_gbp = ccy.index("GBP")
A_diag_eur = res_3ccy.A0[i_eur, i_eur]
A_cross = res_3ccy.A0[i_eur, i_gbp]
print(f"\n|A[EUR,GBP]| / A[EUR,EUR] = {abs(A_cross) / abs(A_diag_eur):.4f}")
print(f"  -> Cross-pair coupling is {abs(A_cross)/abs(A_diag_eur)*100:.1f}% of diagonal")

# Compare EUR/USD QoIs: 2-ccy vs 3-ccy
print("\n=== EUR/USD QoIs: 2-ccy vs 3-ccy ODE ===")
y_flat_2 = np.zeros(2)
y_flat_3 = np.zeros(3)
y_long_2 = np.array([0.0, 10.0])
y_long_3 = np.array([0.0, 10.0, 0.0])

comparisons = [
    ("delta*(y=0) tier1 [bps]",
     res_2ccy.markup(0, "EUR", "USD", 1.0, y_flat_2) / BP,
     res_3ccy.markup(0, "EUR", "USD", 1.0, y_flat_3) / BP),
    ("delta*(y=0) tier2 [bps]",
     res_2ccy.markup(1, "EUR", "USD", 1.0, y_flat_2) / BP,
     res_3ccy.markup(1, "EUR", "USD", 1.0, y_flat_3) / BP),
    ("inv skew [bps]",
     (res_2ccy.markup(0, "EUR", "USD", 1.0, y_long_2) - res_2ccy.markup(0, "EUR", "USD", 1.0, y_flat_2)) / BP,
     (res_3ccy.markup(0, "EUR", "USD", 1.0, y_long_3) - res_3ccy.markup(0, "EUR", "USD", 1.0, y_flat_3)) / BP),
    ("hedge rate [M$/day]",
     res_2ccy.hedge_rate("EUR", "USD", y_long_2),
     res_3ccy.hedge_rate("EUR", "USD", y_long_3)),
]

print(f"  {'QoI':>25s}  {'2-ccy':>12s}  {'3-ccy':>12s}  {'rel diff':>10s}")
for label, v2, v3 in comparisons:
    rd = abs(v2 - v3) / (abs(v3) + 1e-30)
    print(f"  {label:>25s}  {v2:12.6f}  {v3:12.6f}  {rd:10.2e}")

### LHS results: multi-parameter error distribution

In [ ]:
# Compare LHS errors to OAT errors
def collect_errors(data_dict):
    """Collect relative and absolute errors."""
    rel_errs = np.zeros((len(data_dict), N_QOIS))
    abs_errs = np.zeros((len(data_dict), N_QOIS))
    names = sorted(data_dict.keys())
    for i, name in enumerate(names):
        d = data_dict[name]
        for q in range(N_QOIS):
            ode, pde = d['ode_qois'][q], d['pde_qois'][q]
            rel_errs[i, q] = abs(ode - pde) / (abs(pde) + 1e-30)
            abs_errs[i, q] = abs(ode - pde)
    return names, rel_errs, abs_errs

oat_names, oat_rel, oat_abs = collect_errors(oat_data)
lhs_names, lhs_rel, lhs_abs = collect_errors(lhs_data)

print("Relative error comparison: OAT vs LHS")
print(f"{'QoI':>20s}  {'OAT max':>10s}  {'LHS max':>10s}  {'LHS median':>10s}  {'LHS 95%':>10s}")
for q in range(N_QOIS):
    print(f"{QOI_NAMES[q]:>20s}  {oat_rel[:, q].max():10.2e}  {lhs_rel[:, q].max():10.2e}  "
          f"{np.median(lhs_rel[:, q]):10.2e}  {np.percentile(lhs_rel[:, q], 95):10.2e}")

# Histogram of LHS relative errors
fig, axes = plt.subplots(1, N_QOIS, figsize=(4 * N_QOIS, 3.5))
for q in range(N_QOIS):
    ax = axes[q]
    ax.hist(lhs_rel[:, q], bins=10, color='coral', alpha=0.7, edgecolor='white')
    ax.axvline(oat_rel[:, q].max(), color='steelblue', linestyle='--', linewidth=1.5,
               label=f'OAT max ({oat_rel[:, q].max():.2e})')
    ax.set_xlabel('Relative error')
    ax.set_title(QOI_NAMES[q], fontsize=10)
    ax.legend(fontsize=7)

plt.suptitle("LHS Relative Error Distribution", fontsize=13)
plt.tight_layout()
plt.savefig('report/figures/fig_pde_lhs_errors.pdf', bbox_inches='tight', dpi=150)
plt.show()